# Expo Memory Synth: بناء المشهد 3D من فيديوهات وصور

هذا النوتبوك ياخذ مجلد فيه فيديوهات و/أو صور لنفس المكان، ويطلع منه ملف Gaussian Splatting (`.ply`) تعرضونه بالموقع. يشتغل على **Google Colab** و **Kaggle**.

**الخطوات:** فيديوهات وصور ← فريمات ← (طمس الوجيه) ← COLMAP يحسب أماكن الكاميرا ← تدريب gsplat ← ملف `.ply`

**قبل ما تبدين:**
- **Colab:** من القائمة `Runtime > Change runtime type` واختاري **T4 GPU**.
- **Kaggle:** من اللوحة اليمين `Settings`: خلي **Accelerator** على **GPU T4 x2**، وشغلي **Internet** (يحتاج توثيق الحساب برقم الجوال مرة وحدة).
- شغلي الخلايا **وحدة وحدة بالترتيب**، لا تستخدمين `Run all`.
- خلي الجهاز على الشاحن وما ينام (Sleep)، لأن الجلسة تنقطع إذا نام الجهاز.
- **أضمن طريقة (Kaggle بالخلفية):** بـ Kaggle اضغطي **Save Version ← Save & Run All (Commit)**. النوتبوك يشتغل كامل على سيرفرات Kaggle **حتى لو سكرتي الجهاز أو انقطع النت**، ولما يخلص تنزلين الملف من تبويب **Output**.
- **Colab** يقفل جلسات الـ GPU اللي ما تستخدم الـ GPU فترة طويلة (COLMAP يشتغل على المعالج). النوتبوك يحفظ تقدم COLMAP بالـ Drive، فلو انقطعت شغلي الخلايا 1 إلى 4 من جديد وخلية COLMAP تكمل من وين وقفت.
- **إذا خلصت حصة الـ GPU بـ Colab** (رسالة `Cannot connect to GPU backend`): اختاري `Connect without GPU` وسوي الخطوات لين COLMAP (ما يحتاج GPU)، والنتيجة تنحفظ بالـ Drive. لما ترجع الحصة استخدمي خلية **الاسترجاع** قبل التدريب. أو كملي على Kaggle.

## 1. تأكدي إن فيه GPU
لازم يطلع **Tesla T4**. إذا بتسوين COLMAP بس بدون GPU، عادي تطلع هنا رسالة خطأ.

In [ ]:
!nvidia-smi -L

## 2. الإعدادات والمصادر
حطي **كل الفيديوهات والصور** لنفس المكان بمجلد واحد بالـ Drive (من نفس اليوم أفضل). النوتبوك يجمعها كلها بمشهد واحد.
- `SOURCE`: اسم المجلد بالـ Drive. **Colab** يدوره داخل My Drive.
- `FOLDER_ID`: رقم المجلد من رابط المشاركة (الجزء بعد `folders/`). يستخدمه **Kaggle**، أو Colab إذا ما لقى المجلد. لازم يكون مشارك "أي شخص معه الرابط".
- `SCENE_NAME`: اسم مجلد النتايج. غيريه لكل مشهد عشان النتايج ما تختلط.
- `ONLY`: أسماء الملفات اللي تبينها من المجلد بس (مثلاً أطول فيديوهين). فاضية `[]` = كل الملفات. ملفات أقل ومترابطة أحياناً تعطي مشهد أوضح من ملفات كثيرة ما يقدر COLMAP يربطها.
- **صور اللوحات الإرشادية:** حطيها بمجلد فرعي اسمه `labels` داخل المجلد. النوتبوك يتخطاها (ما تساعد بالبناء وتبطئ COLMAP)، وتستخدمينها بعدين مع زر ✨ بالموقع.

In [ ]:
import os, subprocess, glob
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
print('Platform:', 'Kaggle' if ON_KAGGLE else 'Colab')

SOURCE     = 'MiniCorner'                          # <-- مجلد الفيديوهات والصور
FOLDER_ID  = '1EHMBeCovLVrTbPrBfw5F9NImFcUonDSL'   # <-- رقم المجلد من رابطه
SCENE_NAME = 'corner'                               # <-- اسم المشهد (جديد لكل محاولة عشان القديم ما ينمسح)
ONLY       = []      # <-- الملفات المستخدمة بس. [] = كل اللي بالمجلد

if ON_KAGGLE:
    WORK = '/kaggle/working'                      # ينحفظ مع النوتبوك، ويتنزل من لوحة Output
    OUT  = f'{WORK}/outputs/{SCENE_NAME}'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    MYDRIVE = '/content/drive/MyDrive'
    WORK = '/content'
    OUT  = f'{MYDRIVE}/expo-memory-synth/outputs/{SCENE_NAME}'   # بالـ Drive عشان ما تضيع لو انقطعت الجلسة
SCENE = f'{WORK}/{SCENE_NAME}'
os.makedirs(SCENE, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

def find_folder(root, name, depth=3):
    for dirpath, dirnames, _ in os.walk(root):
        if dirpath[len(root):].count(os.sep) >= depth:
            dirnames[:] = []
        if os.path.basename(dirpath) == name:
            return dirpath
    return None

SRC = None if ON_KAGGLE else find_folder(MYDRIVE, SOURCE)
if SRC is None and FOLDER_ID:
    SRC = f'/tmp/{SOURCE}'
    if not glob.glob(f'{SRC}/**/*.*', recursive=True):
        !pip -q install gdown
        !gdown --folder {FOLDER_ID} -O "{SRC}" --remaining-ok

VIDEO_EXT = ('.mov', '.mp4', '.m4v')
PHOTO_EXT = ('.jpg', '.jpeg', '.png', '.heic', '.heif')
files  = sorted(glob.glob(f'{SRC}/**/*.*', recursive=True)) if SRC else []
SKIP_FOLDERS = ('labels', 'label', 'لوحات')   # sign photos: for the hotspot descriptions, not the 3D build
skipped = [f for f in files if any(part.lower() in SKIP_FOLDERS for part in os.path.relpath(f, SRC).split(os.sep)[:-1])]
files  = [f for f in files if f not in skipped]
if ONLY:
    files = [f for f in files if os.path.basename(f) in ONLY]
    missing = [n for n in ONLY if n not in {os.path.basename(f) for f in files}]
    if missing: print('ما لقيت:', missing)
videos = [f for f in files if f.lower().endswith(VIDEO_EXT)]
photos = [f for f in files if f.lower().endswith(PHOTO_EXT)]
print('Source:', SRC)
for v in videos:
    print(f'  video {os.path.basename(v)} ({os.path.getsize(v)/1e6:.0f} MB)')
print(f'{len(videos)} videos, {len(photos)} photos' + (f' ({len(skipped)} files in labels/ skipped)' if skipped else ''))
if not videos and not photos:
    print('ما لقيت شي: شيكي اسم المجلد SOURCE أو رقمه FOLDER_ID')

## 3. تثبيت الأدوات (3-5 دقايق)
- **ffmpeg** لتقطيع الفيديو، و**COLMAP** لحساب أماكن الكاميرا، و**gsplat** للتدريب (إصدار 1.5.3).
- الأخطاء الحمراء عن `numpy` بآخر التثبيت طبيعية، المهم يطلع `COLMAP` و `done`.
- **إذا طلب Restart session:** اختاري **Restart session** بس (مو Disconnect and delete runtime)، وبعدين شغلي خلية 2 من جديد وكملي من خلية 4.
- بدون GPU تتثبت أدوات COLMAP بس.

In [ ]:
import torch
HAS_GPU = torch.cuda.is_available()
GSPLAT_VERSION = '1.5.3'
!apt-get -qq update && apt-get -qq install -y ffmpeg colmap > /dev/null
if HAS_GPU:
    !pip -q install gsplat=={GSPLAT_VERSION}
    !rm -rf {WORK}/gsplat
    !git clone -q --depth 1 --branch v{GSPLAT_VERSION} https://github.com/nerfstudio-project/gsplat.git {WORK}/gsplat
    !pip -q install -r {WORK}/gsplat/examples/requirements.txt
    # examples/datasets has no __init__.py, so the preinstalled `datasets` package would shadow it
    !touch {WORK}/gsplat/examples/datasets/__init__.py
    # gsplat pins numpy<2, which breaks the preinstalled TensorFlow; the trainer only needs tensorboard
    !pip -q uninstall -y tensorflow
    # Compile gsplat's CUDA code now (5-10 min), once and on one GPU. Building it inside the trainer
    # failed on Kaggle ("getcwd() failed"), which lost the whole run after COLMAP.
    print('compiling gsplat CUDA… (5-10 min, no output until it finishes)')
    !cd {WORK} && CUDA_VISIBLE_DEVICES=0 python -c "from gsplat.cuda._backend import _C; assert _C is not None; print('gsplat CUDA ready')"
else:
    print('ما فيه GPU: ثبتت أدوات COLMAP بس. التدريب يحتاج جلسة GPU.')
!colmap -h | head -1
print('done')

## 4. الصور من كل المصادر
- الصور تنحول لـ JPG (حتى صور الآيفون HEIC) ويتعدل اتجاهها.
- من كل فيديو ياخذ صور حسب طوله، عشان المجموع يطلع قريب من `MAX_IMAGES`.
- **`MAX_IMAGES = 900`**: صور أكثر من كل ثانية عشان COLMAP ما يضيع لما تلفين بالكاميرا (قلتها بـ hall3 خلت نص الممر ينطوي). يطول COLMAP لـ 5-6 ساعات، فشغليه بالخلفية على Kaggle.
- كل فيديو بمجلد لحاله (`v1`، `v2`...) والصور بمجلد `photos`، عشان COLMAP يعرف إن كل فيديو كاميرا وحدة.

In [ ]:
MAX_IMAGES = 550
MAX_PHOTOS = 150      # إذا الصور أكثر من كذا ينوخذ منها عينة
MAX_FPS    = 3
LONG_SIDE  = 1600     # أطول ضلع للصورة بالبكسل

import math, shutil
!pip -q install pillow-heif
from PIL import Image, ImageOps
import pillow_heif
pillow_heif.register_heif_opener()

IMAGES = f'{SCENE}/images'
shutil.rmtree(IMAGES, ignore_errors=True)

# Photos
use = photos if len(photos) <= MAX_PHOTOS else photos[::math.ceil(len(photos) / MAX_PHOTOS)]
n_photos = 0
if use:
    os.makedirs(f'{IMAGES}/photos', exist_ok=True)
for p in use:
    try:
        im = ImageOps.exif_transpose(Image.open(p)).convert('RGB')
        im.thumbnail((LONG_SIDE, LONG_SIDE))
        im.save(f'{IMAGES}/photos/{os.path.splitext(os.path.basename(p))[0]}.jpg', quality=92)
        n_photos += 1
    except Exception as e:
        print('skipped', os.path.basename(p), e)
print(f'photos: {n_photos}')

# Videos: share what's left of the budget by length
def duration(v):
    out = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', v],
                         capture_output=True, text=True).stdout.strip()
    return float(out or 0)
durations = {v: duration(v) for v in videos}
total = sum(durations.values()) or 1
budget = max(0, MAX_IMAGES - n_photos)
scale = f"scale='if(gt(iw,ih),min({LONG_SIDE},iw),-2)':'if(gt(iw,ih),-2,min({LONG_SIDE},ih))'"
for i, v in enumerate(videos, 1):
    d = durations[v]
    if d <= 0:
        print('skipped', os.path.basename(v)); continue
    fps = round(min(MAX_FPS, budget / total), 3)
    folder = f'{IMAGES}/v{i}'
    os.makedirs(folder, exist_ok=True)
    !ffmpeg -hide_banner -loglevel error -i "{v}" -vf "fps={fps},{scale}" -q:v 2 {folder}/frame_%04d.jpg
    print(f'v{i} = {os.path.basename(v)}: {d:.0f} s -> {len(os.listdir(folder))} frames')

n = len(glob.glob(f'{IMAGES}/**/*.jpg', recursive=True))
print(f'{n} images in total')
if n < 60: print('قليل! زيدي MAX_IMAGES أو أضيفي فيديوهات')

### (اختياري) شوفي عينة من الصور
تأكدي إنها مو مقلوبة ولا مغبشة. إذا ألوانها باهتة مرة، غالباً الفيديو مصور بـ HDR.

In [ ]:
from IPython.display import Image, display
import glob
frames = sorted(glob.glob(f'{SCENE}/images/**/*.jpg', recursive=True))
for f in frames[::max(1, len(frames) // 4)][:4]:
    display(Image(f, width=300))

## 5. (اختياري) طمس الوجيه للخصوصية
مطفي افتراضياً: الكاشف البسيط يحسب نقوش الحجر وجيه ويطمسها. شغليه (`BLUR_FACES = True`) بس إذا فيه زوار واضحين بالتصوير.

In [ ]:
BLUR_FACES = False   # the simple detector mistakes carvings on stone for faces

try:   # optional: never stops the notebook (cv2 and numpy versions sometimes clash after installing gsplat)
    if not BLUR_FACES:
        raise RuntimeError('off (BLUR_FACES = False)')
    import cv2, glob
    det = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    total = 0
    for f in sorted(glob.glob(f'{SCENE}/images/**/*.jpg', recursive=True)):
        img = cv2.imread(f)
        faces = det.detectMultiScale(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), 1.1, 8, minSize=(60, 60))   # strict: fewer false hits
        for (x, y, w, h) in faces:
            img[y:y+h, x:x+w] = cv2.GaussianBlur(img[y:y+h, x:x+w], (51, 51), 0)
        if len(faces):
            cv2.imwrite(f, img); total += len(faces)
    print('faces blurred:', total)
except Exception as e:
    print('skipped face blurring:', e)

## 6. COLMAP: حساب مكان الكاميرا لكل صورة
هذي أهم خطوة، وتشتغل على المعالج (CPU) مو الـ GPU، فطبيعي Colab يقول إنك ما تستخدمين الـ GPU. سكري التنبيه بس.

- **~350 صورة:** ساعة لساعتين. **~550 صورة (فيديوهات وصور):** ثلاث ساعات تقريباً.
- كل فيديو ينحسب ككاميرا وحدة، وكل صورة ككاميرا لحالها (الآيفون يبدل العدسات).
- **loop detection** يربط الأماكن اللي تتكرر بين الفيديوهات والصور، فيطلع مشهد واحد.
- **يحفظ تقدمه** بعد المطابقة (أطول جزء) وبعد البناء. لو انقطعت الجلسة: شغلي الخلايا 1 إلى 4، وهذي الخلية تكمل من وين وقفت.
- **بـ Kaggle:** كل تشغيل جديد يبدأ فاضي. عشان يكمل من تقدم تشغيل سابق: **Add Input ← Your Work** واختاري نفس النوتبوك (الإصدار السابق). الخلية تلقى مجلد `colmap` حقه وتكمل منه بدال ما تعيد ساعات.
- عشان تتابعين وقت المطابقة، بالـ **Terminal**:
  `python3 -c "import sqlite3; print(sqlite3.connect('/content/corner/database.db').execute('select count(*) from matches').fetchone()[0])"`
  ووقت البناء: `grep "Registering image" /tmp/mapper.log | tail -1`

In [ ]:
import shutil
os.environ['QT_QPA_PLATFORM'] = 'offscreen'   # no display on Colab/Kaggle
IMAGES = f'{SCENE}/images'
DB = f'{SCENE}/database.db'
CKPT = f'{OUT}/colmap'                        # progress saved here (Drive on Colab)
os.makedirs(CKPT, exist_ok=True)
if ON_KAGGLE and not os.path.exists(f'{CKPT}/stage.txt'):
    # an earlier version's output added as Input: /kaggle/input/<notebook>/outputs/<scene>/colmap
    prev = glob.glob(f'/kaggle/input/**/outputs/{SCENE_NAME}/colmap/stage.txt', recursive=True)
    if prev:
        print('found earlier COLMAP progress:', os.path.dirname(prev[0]))
        shutil.rmtree(CKPT, ignore_errors=True)
        shutil.copytree(os.path.dirname(prev[0]), CKPT)
stage = open(f'{CKPT}/stage.txt').read().strip() if os.path.exists(f'{CKPT}/stage.txt') else ''

def save(name, with_images=False):
    shutil.copy(DB, f'{CKPT}/database.db')
    if with_images:
        shutil.rmtree(f'{CKPT}/images', ignore_errors=True)
        shutil.copytree(IMAGES, f'{CKPT}/images')
    open(f'{CKPT}/stage.txt', 'w').write(name)
    print(f'saved progress: {name}')

if stage in ('matched', 'mapped'):
    print(f'resuming from saved progress ({stage})')
    shutil.rmtree(IMAGES, ignore_errors=True)
    shutil.copytree(f'{CKPT}/images', IMAGES)      # the exact images the saved matches belong to
    shutil.copy(f'{CKPT}/database.db', DB)
else:
    !rm -f {DB}
    # One camera per video folder; every photo its own camera
    for folder in sorted(os.listdir(IMAGES)):
        names = sorted(f'{folder}/{f}' for f in os.listdir(f'{IMAGES}/{folder}'))
        with open('/tmp/list.txt', 'w') as fh:
            fh.write('\n'.join(names))
        single = 0 if folder == 'photos' else 1
        print(f'features: {folder} ({len(names)} images)')
        !colmap feature_extractor --database_path {DB} --image_path {IMAGES} --image_list_path /tmp/list.txt \
            --ImageReader.camera_model OPENCV --ImageReader.single_camera {single} --SiftExtraction.use_gpu 0 > /tmp/features.log 2>&1

    # overlap 20: each frame is matched with the next 20, so turns don't break the chain.
    # min_num_inliers 30 (default 15): two images only count as the same spot with strong evidence, so look-alike
    # walls and showcases on the two sides of the corridor are not joined by mistake.
    match = ['colmap', 'sequential_matcher', '--database_path', DB,
             '--SiftMatching.use_gpu', '0', '--SequentialMatching.overlap', '20',
             '--SiftMatching.min_num_inliers', '30']
    loop = []
    VOCAB = '/tmp/vocab_tree_flickr100K_words32K.bin'
    if not os.path.exists(VOCAB):
        !wget -q -O {VOCAB} https://github.com/colmap/colmap/releases/download/3.11.1/vocab_tree_flickr100K_words32K.bin
    if os.path.exists(VOCAB) and os.path.getsize(VOCAB) > 1e6:
        # every 3rd image is compared with the 30 most similar ones anywhere: joins the videos and photos
        loop = ['--SequentialMatching.loop_detection', '1', '--SequentialMatching.vocab_tree_path', VOCAB,
                '--SequentialMatching.loop_detection_period', '3', '--SequentialMatching.loop_detection_num_images', '30']
    print('loop detection:', bool(loop), '| matching… (no output until it finishes)')
    if subprocess.run(match + loop, capture_output=True).returncode != 0 and loop:
        print('loop detection ما اشتغل، أعيد المطابقة بدونه')
        subprocess.run(match, check=True, capture_output=True)
    save('matched', with_images=True)

FAST_MAPPER = False   # True = faster on big scenes, but it can split the scene into separate parts

if stage != 'mapped':
    !rm -rf {SCENE}/sparse {SCENE}/sparse_parts && mkdir -p {SCENE}/sparse
    fast = ('--Mapper.ba_global_images_ratio 1.4 --Mapper.ba_global_points_ratio 1.4 '
            '--Mapper.ba_global_max_num_iterations 30') if FAST_MAPPER else ''
    print('mapping… (follow it in the Terminal with: grep "Registering image" /tmp/mapper.log | tail -1)')
    # A new image is only placed when many of its points agree (defaults 30 and 0.25): stops the mapper from
    # dropping part of a walk in the wrong place, which folded hall3 into an L
    strict = '--Mapper.abs_pose_min_num_inliers 50 --Mapper.abs_pose_min_inlier_ratio 0.3'
    !colmap mapper --database_path {DB} --image_path {IMAGES} --output_path {SCENE}/sparse {fast} {strict} > /tmp/mapper.log 2>&1
    !tail -3 /tmp/mapper.log
    shutil.rmtree(f'{CKPT}/sparse', ignore_errors=True)
    shutil.copytree(f'{SCENE}/sparse', f'{CKPT}/sparse')
    save('mapped')
else:
    shutil.rmtree(f'{SCENE}/sparse', ignore_errors=True)
    shutil.copytree(f'{CKPT}/sparse', f'{SCENE}/sparse')
    print('mapping already done: restored it')
n = len(glob.glob(f'{IMAGES}/**/*.jpg', recursive=True))

### تحققي من النتيجة
الخلية تختار أكبر جزء طلع (لو COLMAP قسم المشهد) وتحفظ نسخة احتياطية.
- **Registered قريب من عدد الصور:** ممتاز.
- **أقل من 70%:** غالباً فيه حركة سريعة أو أماكن فاضية بالتصوير. المشهد بيطلع ناقص بس تقدرين تكملين.

In [ ]:
import re, shutil
def analyze(path):
    out = subprocess.run(['colmap', 'model_analyzer', '--path', path], capture_output=True, text=True)
    text = out.stdout + out.stderr
    m = re.search(r'Registered images:\s*(\d+)', text)
    return (int(m.group(1)) if m else 0), text

models = sorted(d for d in os.listdir(f'{SCENE}/sparse') if os.path.isdir(f'{SCENE}/sparse/{d}'))
stats = {m: analyze(f'{SCENE}/sparse/{m}') for m in models}
for m, (k, _) in stats.items():
    print(f'part {m}: {k} / {n} images')
best = max(stats, key=lambda m: stats[m][0])
if best != '0':
    # gsplat reads sparse/0, so the largest part goes there
    os.makedirs(f'{SCENE}/sparse_parts', exist_ok=True)
    shutil.move(f'{SCENE}/sparse/0', f'{SCENE}/sparse_parts/0')
    shutil.move(f'{SCENE}/sparse/{best}', f'{SCENE}/sparse/0')
k, text = stats[best]
print('\n'.join(l for l in text.splitlines() if re.search('Cameras|Images|Registered|Points|reprojection', l)))
if k < 0.7 * n:
    print(f'\nانبنى {k} من {n} صورة بس. المشهد بيطلع ناقص.')

if not ON_KAGGLE:
    !rm -rf "{OUT}/sparse" "{OUT}/images"
    !cp -r {SCENE}/sparse {SCENE}/images "{OUT}/"   # نسخة احتياطية بالـ Drive
    print('Backup saved to Drive:', OUT)

### (Colab بس) استرجاع نتيجة COLMAP من الـ Drive
إذا انقطعت الجلسة بعد COLMAP، أو سويتي COLMAP بدون GPU ورجعتي الحين بجلسة GPU: شغلي خلية 1 و 2 و 3، وبعدين هذي الخلية، وروحي للتدريب مباشرة.

In [ ]:
import shutil
if not os.path.exists(f'{SCENE}/sparse/0') and os.path.exists(f'{OUT}/sparse'):
    shutil.rmtree(SCENE, ignore_errors=True)
    shutil.copytree(f'{OUT}/images', f'{SCENE}/images')
    shutil.copytree(f'{OUT}/sparse', f'{SCENE}/sparse')
print('frames:', len(glob.glob(f'{SCENE}/images/**/*.jpg', recursive=True)), '| sparse:', os.listdir(f'{SCENE}/sparse'))

## 7. تدريب الـ Gaussian Splatting (يحتاج GPU)
- **قطعة وحدة:** `MAX_STEPS = 7000`، حوالي 15-30 دقيقة على T4.
- **جناح كامل:** `MAX_STEPS = 30000`، حوالي ساعتين.
- **`STRATEGY = 'mcmc'`** (الافتراضي هنا): طريقة تدريب تطلع المشهد أوضح وبنقاط طايرة أقل من `default`، وعدد النقاط ما يتعدى `CAP_MAX`.
- أول مرة ممكن يقعد 5-10 دقايق بدون ما يطلع شي (يسوي compile لأكواد CUDA)، هذا طبيعي.
- إذا طلع خطأ `out of memory`: غيري `DATA_FACTOR` لـ `2` (يصغر الصور للنص).

In [ ]:
import torch
assert torch.cuda.is_available(), 'ما فيه GPU: التدريب يحتاج T4'
MAX_STEPS   = 30000
DATA_FACTOR = 1
STRATEGY    = 'mcmc'      # 'mcmc': sharper, fewer floaters. 'default': gsplat's original densification
CAP_MAX     = 3_000_000   # mcmc only: most splats it may use (more = finer detail, bigger file, more GPU memory)
extra = f'--strategy.cap-max {CAP_MAX}' if STRATEGY == 'mcmc' else ''
RESULTS = f'{WORK}/results/{SCENE_NAME}'

%cd {WORK}/gsplat/examples
!CUDA_VISIBLE_DEVICES=0 python simple_trainer.py {STRATEGY} {extra} \
    --data_dir {SCENE} --data_factor {DATA_FACTOR} --result_dir {RESULTS} \
    --max_steps {MAX_STEPS} --save_ply --ply_steps {MAX_STEPS} --disable_viewer
%cd {WORK}
if not glob.glob(f'{RESULTS}/**/*.ply', recursive=True):
    print('التدريب ما طلّع ملف ply: شوفي الخطأ فوق (آخر الأسطر).')

## 8. حفظ ملف `.ply` وتنزيله

In [ ]:
import glob, shutil
plys = sorted(glob.glob(f'{RESULTS}/**/*.ply', recursive=True), key=os.path.getmtime)
if plys:
    final = f'{OUT}/{SCENE_NAME}.ply'
    shutil.copy(plys[-1], final)
    print('Saved:', final, f'({os.path.getsize(final)/1e6:.1f} MB)')
    if ON_KAGGLE:
        from IPython.display import FileLink, display
        display(FileLink(os.path.relpath(final, WORK)))
        print('اضغطي الرابط فوق، أو نزليه من لوحة Output على اليمين.')
    else:
        print('نزليه من Drive: My Drive > expo-memory-synth > outputs >', SCENE_NAME)
else:
    print('ما لقيت ملف ply. شيكي مخرجات خلية التدريب.')

## 9. الخطوة الجاية
1. افتحي الملف بـ **SuperSplat** (https://superspl.at/editor)، احذفي النقاط الطايرة حول الأطراف، وصدريه بـ `File > Export > Compressed PLY` (يصغر الحجم كثير).
2. حطيه بالريبو داخل `web/` باسم `scene.ply` وشغلي الموقع (التفاصيل بالـ README).
3. حطي النقاط التفاعلية من `index.html?edit`.